In [ ]:
from rdkit import Chem                     # v2024.03.5
from rdkit.Chem import MACCSkeys           # v2024.03.5
import pandas as pd                        # v2.2.2
import numpy as np                         # v1.26.4        
from scipy.integrate import trapezoid      # v1.12.0
from joblib import Parallel, delayed       # v1.4.0

chemical_data = pd.read_csv('../data/chemical/SMILES_data.csv')

def maccs_bits(smiles, failed_smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        print(f"Warning: could not parse SMILES '{smiles}'")
        failed_smiles.append(smiles)
        return [None]*166
    bitvec = MACCSkeys.GenMACCSKeys(mol)
    return list(map(int, bitvec.ToBitString()))

failed_smiles = []
maccs_df = chemical_data['SMILES'] \
              .apply(lambda s: maccs_bits(s, failed_smiles) if pd.notna(s) else [None]*166) \
              .apply(pd.Series)

maccs_df.columns = [f"MACCS_{i}" for i in range(167)]

full_chemical_data = pd.concat([chemical_data, maccs_df], axis=1)
full_chemical_data = full_chemical_data.drop(columns=['MACCS_0'])
full_chemical_data.to_csv('../data/chemical/SMILES_with_MACCS.csv', index=False)

In [ ]:
assay_data    = pd.read_csv("../data/HTS/all_assays_merged.csv")
httr_data     = pd.read_feather("../data/HTTr/signature_scores_all_databases.feather")

chemical_httr_merged = full_chemical_data.merge(httr_data, left_on='dsstox_substance_id', right_on='outcome_id', how='left')
chemical_httr_merged = chemical_httr_merged.dropna(subset=[col for col in chemical_httr_merged.columns if col.startswith("MACCS_")])
chemical_httr_merged = chemical_httr_merged.loc[:, chemical_httr_merged.nunique() > 1]
chemical_httr_merged = chemical_httr_merged.drop(columns=['casn', 'dsstox_substance_id', 'SMILES'])

chemical_httr_assay_merged = chemical_httr_merged.merge(assay_data, left_on='outcome_id', right_on='dtxsid', how='left').drop(columns=['dtxsid'])

In [ ]:
def aggregate_one_bag(bag_data, httr_features):
    bag_data_sorted = bag_data.sort_values('dose_level')
    dose = bag_data_sorted['dose_level'].astype(float).values
    dose_norm = dose / dose.max()

    other_cols = [c for c in bag_data.columns if c not in httr_features + ['dose_level', 'sample_dose_cell']]
    for c in other_cols:
        if bag_data[c].dropna().nunique() > 1:
            raise ValueError(f"Column '{c}' varies within the bag; expected constant metadata.")

    agg_results = {}
    for col in bag_data.columns:
        if col not in httr_features and col != 'dose_level':
            agg_results[col] = bag_data[col].iloc[0]

    for feature in httr_features:
        vals = bag_data_sorted[feature].astype(float).values
        mask = ~np.isnan(vals)
        x = dose_norm[mask]
        y = vals[mask]

        if y.size == 0:
            agg_results[f"{feature}_max"] = np.nan
            agg_results[f"{feature}_dose_at_max"] = np.nan
            agg_results[f"{feature}_AUC_neg"] = np.nan
            continue

        # robust max value (median of top-2)
        if y.size == 1:
            max_robust = y[0]
        else:
            k = min(2, y.size)
            idx_sorted = np.argsort(-np.abs(y))
            top_idx = idx_sorted[:k]
            top_vals = y[top_idx]
            max_robust = np.median(top_vals)

        # dose at NON-robust max (absolute max)
        abs_idx_full = int(np.nanargmax(np.abs(vals)))
        dose_at = dose_norm[abs_idx_full]

        agg_results[f"{feature}_max"] = max_robust
        agg_results[f"{feature}_dose_at_max"] = dose_at

        if x.size >= 2:
            y_neg = np.where(y < 0, y, 0.0)
            auc_neg = np.trapz(y_neg, x=x)
        else:
            auc_neg = np.nan
        agg_results[f"{feature}_AUC_neg"] = auc_neg

    return agg_results

def process_bag_batch(bag_batch, httr_features):
    batch_results = []
    for bag_id, bag_data in bag_batch:
        try:
            agg_result = aggregate_one_bag(bag_data, httr_features)
            batch_results.append(agg_result)
        except Exception as e:
            print(f"Error processing bag {bag_id}: {str(e)}")
            continue
    return batch_results

bag_id_cols   = ['outcome_id', 'epa_sample_id', 'cell_type']
bag_groups    = chemical_httr_assay_merged.groupby(bag_id_cols)
bag_data_list = [(bag_id, group) for bag_id, group in bag_groups]

columns = chemical_httr_assay_merged.columns.tolist()

chemical_cols = [col for col in columns if 'MACCS_' in col]
assay_cols    = [col for col in columns if col.startswith("TOX21_")]
metadata_cols = [col for col in columns if col in ['chnm', 'epa_sample_id', 'dose_level', 'cell_type', 'outcome_id', 'sample_dose_cell']]
httr_cols     = [col for col in columns if col not in chemical_cols + assay_cols + metadata_cols]

batch_size = max(1, len(bag_data_list) // 60)

batches = []
for i in range(0, len(bag_data_list), batch_size):
    batch = bag_data_list[i:i + batch_size]
    batches.append(batch)

results = Parallel(n_jobs=60, verbose=1)(
    delayed(process_bag_batch)(batch, httr_cols) 
    for batch in batches
)

all_aggregated_results = []
for batch_results in results:
    all_aggregated_results.extend(batch_results)

aggregated_df = pd.DataFrame(all_aggregated_results).drop(columns=['sample_dose_cell'])
aggregated_df.to_feather('../data/chemical_httr_assay_aggregated.feather')